# Optuna Tuning & Logging

Runs grid optimization logging to MLflow.


## Optuna Hyperparameter Optimization with MLflow Logs


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
import mlflow
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Set up local sqlite experiment tracking database
os.makedirs("research/experiments", exist_ok=True)
mlflow.set_tracking_uri("sqlite:///research/experiments/mlflow.db")
mlflow.set_experiment("Real_Estate_Optimization")

df = pd.read_csv("research/datasets/processed/real_estate_leads_features.csv")
features = ["square_feet", "bedrooms", "bathrooms", "ad_spend_usd", "impressions", "clicks", "ctr", "cpc"]
X = df[features]
y = df["leads_converted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    with mlflow.start_run(nested=True):
        n_estimators = trial.suggest_int("n_estimators", 50, 150)
        max_depth = trial.suggest_int("max_depth", 4, 10)
        
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        
        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        score = np.mean(cross_val_score(clf, X_train, y_train, cv=3, scoring='f1'))
        
        mlflow.log_metric("f1_score", score)
        return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)
print("Optimization finished. Best params:", study.best_params)
